<a href="https://colab.research.google.com/github/GabyPugaBR/AAI2025/blob/main/2_Product_QnA_Agentic_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Models

In [1]:
!pip install -U langchain
!pip install -U pysqlite3
!pip install -U langchain-openai
!pip install -U langchain-chroma
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U pypdf


from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata
import os

# Setup the LLM
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=userdata.get("OpenAI_API"),
    temperature=0
)

# Setup embeddings
embedding = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=userdata.get("OpenAI_API")
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.4 MB/s eta 0:00:00


## Add  Product Pricing function tool

In [2]:
import pandas as pd
from langchain_core.tools import tool

#Load the laptop product pricing CSV into a Pandas dataframe.
product_pricing_df = pd.read_csv("/content/drive/MyDrive/data/Laptop pricing.csv")
print(product_pricing_df)

@tool
def get_laptop_price(laptop_name:str) -> int :
    """
    This function returns the price of a laptop, given its name as input.
    It performs a substring match between the input name and the laptop name.
    If a match is found, it returns the pricxe of the laptop.
    If there is NO match found, it returns -1
    """

    #Filter Dataframe for matching names
    match_records_df = product_pricing_df[
                        product_pricing_df["Name"].str.contains(
                                                "^" + laptop_name, case=False)
                        ]
    #Check if a record was found, if not return -1
    if len(match_records_df) == 0 :
        return -1
    else:
        return match_records_df["Price"].iloc[0]

#Test the tool. Before running the test, comment the @tool annotation
#print(get_laptop_price("alpha"))
#print(get_laptop_price("testing"))

            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2


## Add Product Features Retrieval Tool

In [5]:

__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

from langchain_core.tools import create_retriever_tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load, chunk and index the contents of the product featuers document.
loader=PyPDFLoader("/content/drive/MyDrive/data/Laptop product descriptions.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=256)
splits = text_splitter.split_documents(docs)

#Create a vector store with Chroma
prod_feature_store = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)

get_product_features = create_retriever_tool(
    prod_feature_store.as_retriever(search_kwargs={"k": 1}),
    name="Get_Product_Features",
    description="""
    This store contains details about Laptops. It lists the available laptops
    and their features including CPU, memory, storage, design and advantages
    """
)

#Test the product feature store
print(prod_feature_store.as_retriever().invoke("Tell me about the AlphaBook Pro") )

[Document(id='51a8ee4e-a0a2-4d7b-88cb-e3c1486889a1', metadata={'source': '/content/drive/MyDrive/data/Laptop product descriptions.pdf', 'creationdate': 'D:20241011143914', 'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'page_label': '1', 'page': 0, 'creator': 'PyPDF', 'total_pages': 1}, page_content='Fictional Laptop Descriptions\nAlphaBook Pro\nThe AlphaBook Pro is a sleek ultrabook with a 12th Gen Intel i7 processor, 16GB of DDR4 RAM,\nand a fast 1TB SSD. Ideal for professionals on the go, this laptop offers an impressive blend of\npower and portability.\nGammaAir X\nGammaAir X combines an AMD Ryzen 7 processor with 32GB of DDR4 memory and a 512GB\nNVMe SSD. Its thin and light form factor makes it perfect for users who need high performance in a\nportable design.\nSpectraBook S\nDesigned for power users, SpectraBook S features an Intel Core i9 processor, 64GB RAM, and a\nmassive 2TB SSD. This workstation-class laptop is perfect for intensive tasks like video editing and\n3

## Setup a Product QnA chatbot

In [6]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent

system_prompt_content = """
You are a professional chatbot that answers questions about laptops sold by your company.
To answer questions about laptops, you will ONLY use the available tools and NOT your own memory.
You will handle small talk and greetings by producing professional responses.
"""

tools = [get_laptop_price, get_product_features]
checkpointer = MemorySaver()

product_QnA_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt_content,
    debug=False,
    checkpointer=checkpointer
)


In [7]:
#Setup chatbot
import uuid
#To maintain memory, each request should be in the context of a thread.
#Each user conversation will use a separate thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

#Test the agent with an input
inputs = {"messages":[
                HumanMessage("What are the features and pricing for GammaAir?")
            ]}

#Use streaming to print responses as the agent  does the work.
#This is an alternate way to stream agent responses without waiting for the agent to finish
for stream in product_QnA_agent.stream(inputs, config, stream_mode="values"):
    message=stream["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()

================================ Human Message =================================

What are the features and pricing for GammaAir?
================================== Ai Message ==================================
Tool Calls:
  Get_Product_Features (call_fVJRHoyLI4VDVXRw0rHioWpD)
 Call ID: call_fVJRHoyLI4VDVXRw0rHioWpD
  Args:
    query: GammaAir
  get_laptop_price (call_i73KBGxsFTp3xGDwa72rtEtV)
 Call ID: call_i73KBGxsFTp3xGDwa72rtEtV
  Args:
    laptop_name: GammaAir
================================= Tool Message =================================
Name: get_laptop_price

1399
================================== Ai Message ==================================

The **GammaAir X** features include:

- **Processor**: AMD Ryzen 7
- **Memory**: 32GB of DDR4 RAM
- **Storage**: 512GB NVMe SSD
- **Design**: Thin and light form factor, making it perfect for high performance in a portable design.

The price for the GammaAir X is **$1399**.


Execute the Product QnA Chatbot

In [8]:
import uuid
#Send a sequence of messages to chatbot and get its response
#This simulates the conversation between the user and the Agentic chatbot
user_inputs = [
    "Hello",
    "I am looking to buy a laptop",
    "Give me a list of available laptop names",
    "Tell me about the features of  SpectraBook",
    "How much does it cost?",
    "Give me similar information about OmegaPro",
    "What info do you have on AcmeRight ?",
    "Thanks for the help"
]

#Create a new thread
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

for input in user_inputs:
    print(f"----------------------------------------\nUSER : {input}")
    #Format the user message
    user_message = {"messages":[HumanMessage(input)]}
    #Get response from the agent
    ai_response = product_QnA_agent.invoke(user_message,config=config)
    #Print the response
    print(f"AGENT : {ai_response['messages'][-1].content}")


----------------------------------------
USER : Hello
AGENT : Hello! How can I assist you today? If you have any questions about our laptops, feel free to ask!
----------------------------------------
USER : I am looking to buy a laptop
AGENT : That's great! What specific features or specifications are you looking for in a laptop? For example, do you have preferences regarding the CPU, memory, storage, or design?
----------------------------------------
USER : Give me a list of available laptop names
AGENT : Here are some of the available laptops:

1. **AlphaBook Pro** - A sleek ultrabook with a 12th Gen Intel i7 processor, 16GB of DDR4 RAM, and a fast 1TB SSD.

2. **GammaAir X** - Combines an AMD Ryzen 7 processor with 32GB of DDR4 memory and a 512GB NVMe SSD.

3. **SpectraBook S** - Features an Intel Core i9 processor, 64GB RAM, and a massive 2TB SSD, ideal for intensive tasks.

4. **OmegaPro G17** - A gaming powerhouse with a Ryzen 9 5900HX CPU, 32GB RAM, and a 1TB SSD, designed for

In [9]:
#conversation memory by user
def execute_prompt(user, config, prompt):
    inputs = {"messages":[("user",prompt)]}
    ai_response = product_QnA_agent.invoke(inputs,config=config)
    print(f"\n{user}: {ai_response['messages'][-1].content}")

#Create different session threads for 2 users
config_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}
config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

#Test both threads
execute_prompt("USER 1", config_1, "Tell me about the features of  SpectraBook")
execute_prompt("USER 2", config_2, "Tell me about the features of  GammaAir")
execute_prompt("USER 1", config_1, "What is its price ?")
execute_prompt("USER 2", config_2, "What is its price ?")



USER 1: The **SpectraBook S** is designed for power users and boasts impressive features, including:

- **Processor**: Intel Core i9
- **Memory**: 64GB RAM
- **Storage**: 2TB SSD

This workstation-class laptop is ideal for intensive tasks such as video editing and 3D rendering, providing the performance needed for demanding applications.

USER 2: The **GammaAir X** is a high-performance laptop that features:

- **Processor**: AMD Ryzen 7
- **Memory**: 32GB of DDR4 RAM
- **Storage**: 512GB NVMe SSD
- **Design**: Thin and light form factor, making it ideal for users who need portability without sacrificing performance.

This laptop is perfect for those who require a powerful machine for various tasks while on the go.

USER 1: The price of the **SpectraBook S** is **$2499**.

USER 2: The price of the **GammaAir X** is $1399.
